# Exploring the UK Parliament Written Questions API

In this notebook you'll learn how to **interact with a public API** using Python and JSON.

We'll use the UK Parliament’s open **Written Questions & Answers API**, which publishes information about parliamentary questions asked by MPs and peers.

### Content of the notebook:
- What an API and JSON are
- Making a request to a public API with `requests`
- Understanding endpoints and parameters
- inspecting the data

For this API, no key is required — everything here is public data from the UK Parliament.


In [84]:
import json
from pathlib import Path
from datetime import date, timedelta

import requests
import pandas as pd

pd.set_option("display.max_columns", None)

## JSON

APIs usually send and receive **JSON** data.

JSON is just text made up of:
- **Objects** – dictionaries in Python, written as `{ "key": "value" }`
- **Arrays** – lists in Python, written as `[ ... ]`

Example:
```json
{
  "name": "Alice",
  "age": 30,
  "skills": ["Python", "Data"]
}


## APIs

Every API has a few basic parts:

| Term | Meaning | Example |
|------|----------|----------|
| **Base URL** | The root address of the API | `https://questions-statements-api.parliament.uk` |
| **Endpoint** | A specific dataset or function | `/api/writtenquestions/questions` |
| **HTTP method** | Action type (most often `GET`) | `GET /api/writtenquestions/questions` |
| **Parameters** | Filters or options after a `?` | `?searchTerm=energy&page=1&pageSize=20` |
| **Status codes** | Indicate success or errors | `200 OK`, `400 Bad Request`, `404 Not Found` |

We’ll start with the **Written Questions** endpoint, which returns recent questions asked in the House of Commons or Lords.  

For reference, you can explore related APIs and endpoints on the official documentation pages:
- **Written Questions & Statements API:** [https://questions-statements-api.parliament.uk](https://questions-statements-api.parliament.uk)  
- **Members API (for MP information):** [https://members-api.parliament.uk/index.html](https://members-api.parliament.uk/index.html)


In [93]:
import requests
import pandas as pd

# 1. Define the base URL and endpoint
BASE_URL = "https://questions-statements-api.parliament.uk"
ENDPOINT = "/api/writtenquestions/questions"

# 2. Build the full URL
url = BASE_URL + ENDPOINT

# 3. Set fixed query parameters
params = {
    "searchTerm": "quantum poetry",       # keyword
    "fromDate": "2025-09-01",           # start date (YYYY-MM-DD)
    "toDate":   "2025-11-06",           # end date          
    "page":     1,
    "pageSize": 20
}

print(f"Requesting this url: {url}")
print(f"With these params: {params}")

# 4. Make the API call
try:
    response = requests.get(url, params=params, headers={"Accept": "application/json"}, timeout=15)
    response.raise_for_status()
    data = response.json()
    print("Status:", response.status_code)
except Exception as e:
    print("Unexpected error:", e)
    data = None

Requesting this url: https://questions-statements-api.parliament.uk/api/writtenquestions/questions
With these params: {'searchTerm': 'quantum poetry', 'fromDate': '2025-09-01', 'toDate': '2025-11-06', 'page': 1, 'pageSize': 20}
Status: 200


### Understanding `page`, `pageSize`, and Pagination in the API Response

When you make an API request to the UK Parliament Questions API, you don’t always get *all* the results at once — the data is **paginated**.  
That means the full dataset is split into smaller, more manageable chunks called **pages**.  
This keeps the response quick and prevents overloading your computer or the API server.

---

#### 🧭 Key Pagination Parameters

- **`page`** — tells the API *which page of results* you want to retrieve.  
  - `page=1` means “give me the first page.”  
  - `page=2` means “give me the second page.”  
  - If you loop through all pages, you can collect the complete dataset.

- **`pageSize`** — defines *how many items* should appear on each page.  
  - Example: `pageSize=20` → each page will contain **up to 20 records**.  
  - Increasing this number means more data per request, but also larger responses.

---

#### ⚙️ The Request in This Example

Here’s the code we used:

```python
params = {
    "searchTerm": "quantum poetry",
    "fromDate": "2025-09-01",
    "toDate": "2025-11-06",
    "page": 1,
    "pageSize": 20
}
```

This means:
- We searched for written questions containing the search term **“quantum poetry”**  
- Between **1 September 2025** and **6 November 2025**  
- We asked for **page 1** (the first set of results)  
- With **20 results per page**

---

#### 📦 What the API Returned

The response included:

```json
"totalResults": 273
```

That means there are **273 total written questions** matching our filters.

However, because we set `pageSize` to **20**, this **first API call only returned 20 results** — the first “slice” of the full dataset.



In [94]:
print(f"The number of TOTAL results is {data['totalResults']}")
print(f"As we set page to 1, and pageSize to 20, the number of results that we have now retrieved is {len(data['results'])}")

The number of TOTAL results is 273
As we set page to 1, and pageSize to 20, the number of results that we have now retrieved is 20


### 🔁 Looping through all pages (== getting all the data)

When working with paginated APIs, there are **two common ways** to decide when to stop your loop:

1. **Stop when the API returns no results** (simplest method).  
2. **Stop when you’ve reached the last page**, calculated from `totalResults` and `pageSize`.

In this example, we use **both checks** — so the loop stops safely no matter what the API does.

**Note:**  
This particular API does not accept `page` and `pageSize` directly, so the code converts them into `skip` and `take` internally.  
Don’t worry — you can still think of it as normal page-based pagination.

---

#### Explanation

- The first request fetches page 1 (the first 20 results, since `pageSize=20`).
- The API response includes:
  - `results`: the actual list of questions  
  - `totalResults`: the total number of matching questions in the system  
- The code uses `math.ceil(totalResults / pageSize)` to calculate how many pages there are in total.
- Each loop request increments the page number (`page=2`, `page=3`, …).
- The loop stops automatically when:
  - the API returns no results (end reached early), **or**
  - the current page number equals the total number of pages.

This makes the loop robust — it always stops, even if the API behaves unexpectedly.

---

#### 🧠 How do you know the `pageSize`?

- Most APIs (like this one) **let you set your own** `pageSize` — the number of items per page.
- The API might have a **maximum allowed value**, such as 20, 50, or 100.
- If you request more than allowed, it will usually:
  - cap the value automatically, or  
  - return an error message.

In this example:

```python
"pageSize": 20

```
means “please send 20 results per request.”

---

> **Summary:**  
> This `while` loop keeps fetching pages of data, increasing the page number each time,  
> and stops cleanly when there’s nothing left to fetch — or when the last page is reached.


In [1]:
import math, requests

url = "https://questions-statements-api.parliament.uk/api/writtenquestions/questions"

params = {

    "searchTerm": "quantum poetry",
    "fromDate": "2025-09-01",
    "toDate": "2025-11-06",
    "page": 1,      # which page we want
    "pageSize": 20  # how many results per page
}

all_results = []

while True:
    # Convert page/pageSize → skip/take (the API needs these)
    skip = (params["page"] - 1) * params["pageSize"]
    take = params["pageSize"]

    # Ask the API for this "page"
    r = requests.get(url, params={
        "searchTerm": params["searchTerm"],
        "fromDate": params["fromDate"],
        "toDate": params["toDate"],
        "skip": skip,
        "take": take
    })
    data = r.json()

    results = data.get("results", [])
    if not results:
        break

    all_results.extend(results)
    print(f"Page {params['page']} → {len(results)} results")

    # Stop when we're on the last page
    total = data.get("totalResults", 0)
    total_pages = math.ceil(total / params["pageSize"])
    if params["page"] >= total_pages:
        break

    params["page"] += 1

print(f"\nCollected {len(all_results)} results.")


Page 1 → 20 results
Page 2 → 20 results
Page 3 → 20 results
Page 4 → 20 results
Page 5 → 20 results
Page 6 → 20 results
Page 7 → 20 results
Page 8 → 20 results
Page 9 → 20 results
Page 10 → 20 results
Page 11 → 20 results
Page 12 → 20 results
Page 13 → 20 results
Page 14 → 13 results

Collected 273 results.


In [2]:
print(all_results[0])

{'value': {'id': 1848185, 'askingMemberId': 5100, 'askingMember': None, 'house': 'Commons', 'memberHasInterest': False, 'dateTabled': '2025-11-04T00:00:00', 'dateForAnswer': '2025-11-11T00:00:00', 'uin': '88010', 'questionText': 'To ask the Secretary of State for Business and Trade, pursuant to the Answer to Question 50646 on Trade Agreements: USA, what progress his Department has made on trade talks with the United States.', 'answeringBodyId': 214, 'answeringBodyName': 'Department for Business and Trade', 'isWithdrawn': False, 'isNamedDay': False, 'groupedQuestions': [], 'answerIsHolding': False, 'answerIsCorrection': False, 'answeringMemberId': 1446, 'answeringMember': None, 'correctingMemberId': None, 'correctingMember': None, 'dateAnswered': '2025-11-11T00:00:00', 'answerText': 'The UK continues to engage across the range of issues outlined in the General Terms for the UK-US Economic Prosperity Deal.During President Trump’s State Visit in September, the UK and US announced the Tech

##  Cursor-based pagination

Some APIs don’t use page numbers (`page=1, 2, …`). Instead, they use a **cursor**: a token that marks your position in the dataset. You **start** with a special value (often `*`) and the API returns a **`nextCursor`** you pass back to fetch the next batch. You **stop** when there is no `nextCursor`.

### How it works
1. **Start**: send your first request with `cursor="*"` (means “from the beginning”).
2. **Receive**: the API returns a batch of results **and** some metadata (e.g., `header.nextCursor`).
3. **Continue**: send the **next** request using that `nextCursor`.
4. **Stop**: when the response has **no `nextCursor`**, you’ve reached the end.

> You **do not** need to calculate total pages ahead of time.  
> You **do** need a loop that keeps following the cursor until it disappears.

### What to watch for
- Use consistent, valid date formats (`YYYY-MM-DD`).
- Handle network issues: `timeout`, `raise_for_status()`.
- Decide how you want to store data:
  - **Nested**: append the full JSON response each time (metadata + results).
  - **Flat**: extend a list with only the items (e.g., `data["results"]`).

---

## Example: Cursor loop (to be used if you encounter an API with cursor)

```python
import requests
import pandas as pd

BASE_URL = "xxxxxx"

params = {
    "keywords": "climate change",
    "fromStartDate": "1900-10-01",
    "toEndDate": "2025-11-01",
    "cursor": "*",          # starting point (like page 1)
    "pageSize": 50          # number of results per request
}

print(f"Requesting data from: {BASE_URL}")
print(f"Initial parameters: {params}")

# This will store the *entire JSON responses* (nested structure)
all_data = []

more_pages = True

while more_pages:
    response = requests.get(BASE_URL, params=params, headers={"Accept": "application/json"}, timeout=15)
    response.raise_for_status()
    data = response.json()

    # Append the full page response (metadata + results)
    all_data.append(data)
    print(f"Fetched batch with {len(data.get('results', []))} results for cursor: {params['cursor']}")

    # Get the next cursor for continuation
    next_cursor = data.get("header", {}).get("nextCursor")

    if next_cursor:
        params["cursor"] = next_cursor
        print("Moving to next cursor...")
    else:
        more_pages = False
        print("No more results — stopping loop.")

print(f"\nDone! Collected {len(all_data)} full response objects (nested).")
```

### Access patterns (after the loop)
- First page’s metadata: `all_data[0]["header"]`
- First page’s results list: `all_data[0]["results"]`
- First item of the second page: `all_data[1]["results"][0]` (if it exists)

> **Tip:** If you later want a flat table, you can iterate over `all_data` and concatenate the `results` lists into a single DataFrame.
